# Encontro 3 · Raspando a Aurora Imóveis

**Coleta e Preparação de Dados · PUCRS Online** · prática de web scraping, CSS, XPath e BeautifulSoup

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lucaskup/pucrs-coleta-e-preparacao/blob/main/aula3/pratica/encontro3.ipynb)

Alvo: <https://lucaskup.github.io/pucrs-coleta-e-preparacao/aula3/pratica/imoveis/>, um site **fictício**
com 60 anúncios em 5 páginas, cada um com uma página de detalhe. Abra o site no navegador e use
*Inspecionar* antes de escrever qualquer seletor.

| Etapa | O quê | Tempo |
| --- | --- | --- |
| 1. Missão comum | todos os grupos coletam a listagem completa | 25 min |
| 2. Extensão do grupo | cada grupo recebe uma extensão (A, B, C ou D) | 20 min |
| 3. Revisão cruzada | na apresentação, outro grupo revisa o seu código com o checklist da revisão cruzada | 40 min |

A base completa desse site (300 anúncios) também é usada nas práticas de análise exploratória, visualização e preparação.

In [ ]:
import re
import time
from urllib.parse import urljoin

import pandas as pd
import requests
from bs4 import BeautifulSoup

BASE = "https://lucaskup.github.io/pucrs-coleta-e-preparacao"
SITE = f"{BASE}/aula3/pratica/imoveis/"

sessao = requests.Session()
sessao.headers.update({"User-Agent": "coleta-pucrs/1.0 (seu.email@edu.pucrs.br)"})   # identifique-se!

## 1. Missão comum: a listagem completa (25 min)

Monte um DataFrame com **uma linha por anúncio** e as colunas:

`codigo`, `titulo`, `tipo`, `bairro`, `preco` (float), `area_m2` (float), `quartos` (int), `vagas` (int), `link` (URL absoluta)

Regras do jogo:
- siga a paginação pelo link **próxima**, sem montar `pagina-2`, `pagina-3`… à mão;
- `timeout`, `raise_for_status()` e `time.sleep(1)` entre páginas;
- preços aparecem como `R$ 1.057.000` e áreas como `72,6 m²` (vírgula decimal!); anúncios sem preço
  mostram `Consulte`, que vira ausente (`NaN`);
- o código do anúncio está em um atributo do `<article>`. Qual?

In [ ]:
def numero_br(texto):
    """'R$ 1.057.000' -> 1057000.0 ; '72,6 m²' -> 72.6 ; 'Consulte' -> NaN"""
    ...

assert numero_br("R$ 1.057.000") == 1057000.0
assert numero_br("72,6 m²") == 72.6
assert pd.isna(numero_br("Consulte"))

In [ ]:
linhas, url = [], SITE
while url:
    ...

anuncios = pd.DataFrame(linhas)
print(anuncios.shape)
anuncios.head()

In [ ]:
# conferência rápida (rode depois de terminar)
assert anuncios.shape == (60, 9), anuncios.shape
assert anuncios["codigo"].is_unique
assert anuncios["preco"].isna().sum() == 2       # os dois "Consulte"
assert anuncios["link"].str.startswith("http").all()
print("missão comum ok")

---
## 2. Extensões (20 min): o mediador sorteia uma por grupo

As extensões A e B visitam só os **10 primeiros** anúncios, para poupar o servidor.

### Extensão A: ficha técnica com XPath (lxml)

A ficha de cada imóvel é uma lista de definições: `<dt>Condomínio</dt><dd>R$ 840/mês</dd>`.
Escreva `ler_ficha(url)` usando **XPath** com o eixo `following-sibling` para pegar o valor que vem
depois de cada rótulo. Devolva um dicionário com `codigo`, `condominio`, `iptu`, `andar`, `ano_construcao`
e `conservacao`. Pergunta: o que aparece no condomínio das casas, e como você vai representar isso?

In [ ]:
from lxml import html as lhtml

def ler_ficha(url):
    ...

fichas = pd.DataFrame([ler_ficha(u) for u in anuncios["link"].head(10)])
fichas

### Extensão B: comodidades em colunas binárias

Cada página de detalhe lista as comodidades em `ul.comodidades li`. Para os 10 primeiros anúncios,
monte uma tabela com **uma coluna 0/1 por comodidade**. Pergunta: esse é um atributo binário
simétrico ou assimétrico? Justifique.

In [ ]:
...

### Extensão C: robots.txt e sitemap.xml

1. Leia o `robots.txt` do site com `urllib.robotparser`. Podemos coletar os anúncios? E o `painel-corretor/`?
   Qual intervalo entre requisições o site pede?
2. Leia o `sitemap.xml` (parser `lxml-xml`) e extraia as URLs de anúncios (`/imovel/`).
3. Compare com os links que a missão comum coletou: os dois conjuntos são iguais?

In [ ]:
from urllib.robotparser import RobotFileParser
...

### Extensão D: robustez dos conversores

A função `numero_br` da missão comum foi testada em três casos. Escreva **mais seis** `assert` com
entradas que poderiam aparecer em outros sites de imóveis e corrija a função se alguma falhar.
Sugestões: `"R$ 950 mil"`, `"R$ 1,2 milhão"`, `"1.200"` (é mil e duzentos ou um vírgula dois?),
`"  85m2 "`, `"Isento"`, `"—"`. Decida e **documente** o que cada uma deve devolver.

In [ ]:
...

---
## 3. Checklist da revisão cruzada

Na apresentação, o grupo revisor responde **sim / não / parcialmente** para o código do outro grupo e
aponta a linha:

- [ ] Identifica-se no `User-Agent` e usa `timeout` e `raise_for_status()`?
- [ ] Respeita o intervalo do `robots.txt` (`time.sleep`)?
- [ ] Segue a paginação pelo link, sem URLs escritas à mão?
- [ ] Usa seletores por classe ou atributo, estáveis a mudanças de layout (sem `div:nth-child(3)`)?
- [ ] Resolve links relativos com `urljoin`?
- [ ] Converte `R$ 1.057.000` e `72,6 m²` corretamente e trata `Consulte` como ausente (`NaN`)?
- [ ] O resultado tem 60 linhas e códigos únicos?